# Onnx Runtime on GPU/NE

In [1]:
import torch
from transformers import AutoModelForQuestionAnswering, BertTokenizer

device = torch.device("cuda")
model_id = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)
model.eval()
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
session = onnxruntime.InferenceSession(
    export_model_path,
    sess_options,
    providers=['CUDAExecutionProvider']
)

In [ ]:
# Decoder only model
import torch
from transformers import GPT2Tokenizer, AutoModelForCausalLM

model_id = 'openai-community/gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
device = torch.device("cuda")
model.eval().to(device)

In [ ]:
import os

output_dir = os.path.join(".", "onnx_models")
onnx_model_filename = 'bert-base-uncased.onnx'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
export_model_path = os.path.join(output_dir, onnx_model_filename)

In [ ]:
tokenized_inputs = tokenizer(
    "The story so far: in the beginning, the universe was created.",
    return_attention_mask=False,
    return_tensors="pt"
)
tokenized_inputs.to(device)
inputs_sample = {
    'input_ids':  tokenized_inputs['input_ids']
}

In [ ]:
with torch.no_grad():
    torch.onnx.export(
        model,
        inputs_sample,
        export_model_path,
        export_params=True,
        opset_version=15,
        input_names=['input_ids']
    )

In [ ]:
import onnxruntime
import numpy

session = onnxruntime.InferenceSession(export_model_path, providers=["CUDAExecutionProvider"])
onnx_input_ids = tokenizer(
    "The story so far: in the beginning, the universe was created.",
    return_attention_mask=False,
    return_tensors="np"
)
ort_inputs = {
    "input_ids": onnx_input_ids['input_ids']
}

ort_outputs = session.run(None, ort_inputs)

In [ ]:
from onnxruntime.transformers import optimizer

optimized_model_path = os.path.join(output_dir, 'gpt-2-onnx_opt_gpu.onnx')
optimized_model = optimizer.optimize_model(
    export_model_path,
    model_type='gpt2',
    use_gpu=True,
    num_heads=12,
    hidden_size=768
)
optimized_model.save_model_to_file(optimized_model_path)

In [ ]:
import onnxruntime as rt

sess_options = rt.SessionOptions()

sess_options.graph_optimization_level = rt.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.optimized_model_filepath = "<model_path\optimized_model.onnx>"

session = rt.InferenceSession(None, sess_options)

In [ ]:
import onnxruntime as rt

sess_options = rt.SessionOptions()

sess_options.graph_optimization_level =
➥rt.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.optimized_model_filepath = "<model_path\optimized_model.onnx>"

session = rt.InferenceSession(None, sess_options)

Guglielmo Iozzia. Domain-Specific_Small_Language_Models (Function). Kindle Edition.